### 1.1 Lists and dictionaries

In [ ]:
lines = [
    "INFO user=alice action=login",
    "ERROR user=bob action=upload",
    "INFO user=carol action=logout",
]

lines.append("WARNING user=dave action=retry")
label_counts = {"INFO": 2, "ERROR": 1, "WARNING": 1}    #在NLP里，list存储okens or lines，dict则存储counts, labels, or exceptional word forms.

print("Number of lines:", len(lines))
print("Second line:", lines[1])
print("Error count:", label_counts["ERROR"])    #有一个人的信息是error

Number of lines: 4
Second line: ERROR user=bob action=upload
Error count: 1


### Conditionals and loops

In [ ]:
### 找出list存的用户信息里面有error和warning的
for line in lines:
    if line.startswith("ERROR"):
        print("Needs attention:", line)
    elif line.startswith("WARNING"):
        print("Check soon:", line)
    else:
        print("Normal:", line)

Normal: INFO user=alice action=login
Needs attention: ERROR user=bob action=upload
Normal: INFO user=carol action=logout
Check soon: WARNING user=dave action=retry


### Practice 1 — Basic Python text processing

In [8]:
for line in lines:
    print(line)
print()

def error_usernames(log_lines):
    usernames = []
    for line in log_lines:      #取每条用户信息
        if line.startswith("ERROR"):    #如果信息第一个是error
            for field in line.split():  # 一个一个取这个信息 ERROR user=bob action=upload
                if field.startswith("user="):   #取到了 'user=bob'
                    usernames.append(field.split("=", maxsplit=1)[1])   #只从'='切一次，['user', 'bob']，然后取第二个
    return usernames

print(error_usernames(lines))   # ['bob']

INFO user=alice action=login
ERROR user=bob action=upload
INFO user=carol action=logout
WARNING user=dave action=retry

['bob']


### Regular Language and Expressions 正则表达式

### 2.1 Core regex syntax in Python

| Pattern | Meaning | Example |
|---|---|---|
| `x` | literal character | `cat` |
| `A|B` | alternative | `cat|dog` |
| `[...]` | one character from a set | `[A-Z]` |
| `[^...]` | one character not in a set | `[^0-9]` |
| `.` | almost any character | `a.c` |
| `*` | zero or more | `ab*` |
| `+` | one or more | `ab+` |
| `?` | optional | `colou?r` |
| `{m,n}` | between `m` and `n` repetitions | `\d{2,4}` |
| `(...)` | capturing group | `(ha)+` |
| `\d`, `\w`, `\s` | digit, word character, whitespace | `\d+` |
| `^`, `$` | start and end anchors | `^INFO.*$` |
| `\b` | word boundary | `\bcat\b` |


### 2.2 Importing `re` and choosing a matching function

- `re.fullmatch(pattern, text)` — the entire string must match;
- `re.match(pattern, text)` — match only at the start;
- `re.search(pattern, text)` — find the first match anywhere;
- `re.findall(pattern, text)` — return all non-overlapping matches;
- `re.finditer(pattern, text)` — iterate over match objects; and
- `re.sub(pattern, replacement, text)` — replace matches.

#### 去掉空格并提取文字

In [ ]:
import re

text = "hello   world  ashi"
pattern = (r"\s+")

print(re.split(pattern, text))  # \s+是匹配 1 个或多个空白字符。 输出['hello', 'world', 'ashi']

['hello', 'world', 'ashi']


#### 提取邮箱地址

In [ ]:
#简化版提取邮箱
''' 
\w: 匹配字母、数字、下划线，例如 a、B、3、_
+: 前面的东西出现 1 次或更多次
@: 普通的 @
\.: 匹配真正的句点 .
.: 在正则里本身代表“任意字符”，所以想匹配真正的 .，必须写成 \.

所以pattern拆开是\w+   @   \w+   \.   \w+
也就是abc + @ + gmail + . + com
'''

email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
emails = ["test.email@example.com", "invalid-email.com", "student@xjtlu.edu.cn"]

for email in emails:
    is_valid = re.fullmatch(email_pattern, email) is not None
    print(f"{email:28} -> {is_valid}")      # email:28表示给 email 预留 28 个字符宽度。

test.email@example.com       -> True
invalid-email.com            -> False
student@xjtlu.edu.cn         -> True


In [15]:
#完整版提取邮箱
''' 
[]是匹配括号里列出的任意一个字符。
A-Z      大写字母
a-z      小写字母
0-9      数字
.        点
_        下划线
%        百分号
+        加号
-        减号
'''
email_pattern_without_plus = (
    r"[A-Za-z0-9._%+-][A-Za-z0-9._%+-]*"
    r"@"
    r"[A-Za-z0-9.-][A-Za-z0-9.-]*"
    r"\."
    r"[A-Za-z]{2,}"     # {2,}是前面的东西至少出现 2 次，也就是最少有两个字母
)

for email in emails:
    print(email, bool(re.fullmatch(email_pattern_without_plus, email)))

test.email@example.com True
invalid-email.com False
student@xjtlu.edu.cn True


#### 用re.findall()提取数字

In [ ]:
sentence = "There are 12 cats, 34 dogs, and 56 birds."
numbers_as_text = re.findall(r"\d+", sentence)              #输出为['12', '34', '56']，是字符串
numbers = [int(number) for number in numbers_as_text]       #把字符串里的数字变成真数字

print(numbers_as_text)
print("Total animals:", sum(numbers))

['12', '34', '56']
Total animals: 102


#### re.search()来Capturing groups

In [ ]:
''' 
phone_pattern里面有三个(), re.search() 找到匹配后，会返回一个 Match 对象。
所以phone_match有四个group：
group(0) → 默认整体
group(1) → 第1个 ()
group(2) → 第2个 ()
group(3) → 第3个 ()

'''
phone_text = "My phone number is 123-456-7890."
phone_pattern = r"(\d{3})-(\d{3})-(\d{4})"
phone_match = re.search(phone_pattern, phone_text)

if phone_match:
    print("Full match:", phone_match.group(0))
    print("Area code:", phone_match.group(1))
    print("Local number:", phone_match.group(2), phone_match.group(3))

Full match: 123-456-7890
Area code: 123
Local number: 456 7890


#### 用(?P\<name\>pattern)来命名一个group


In [19]:
''' 
1. r"(?P<date>\d{4}-\d{2}-\d{2})\s+" 
(?P<date>...)：创建一个名字叫 date 的捕获组
\d{4}-\d{2}-\d{2}：匹配日期，如 2026-09-08
\s+：匹配一个或多个空白字符

2. r"(?P<level>INFO|WARNING|ERROR)\s+"
表示日志等级只能是：INFO WARNING ERROR
并把结果保存到level组

'''
log_line = "2026-09-08 ERROR user=bob action=upload"
log_pattern = (
    r"(?P<date>\d{4}-\d{2}-\d{2})\s+"
    r"(?P<level>INFO|WARNING|ERROR)\s+"
    r"user=(?P<user>\w+)\s+"
    r"action=(?P<action>\w+)"
)

log_match = re.fullmatch(log_pattern, log_line)
if log_match:
    print(log_match.groupdict())        #log_match.groupdict()会把所有命名捕获组直接转换成字典

{'date': '2026-09-08', 'level': 'ERROR', 'user': 'bob', 'action': 'upload'}


### Practice 2 — A small NLP application：removing `ing` from `running`, for example, produces `runn`, and context can change a word's lemma.

In [ ]:
'''
输出：
playing    -> play
played     -> play
happily    -> happy
running    -> runn
'''
def simple_suffix_normalize(word):
    word = word.lower()
    word = re.sub(r"ily$", "y", word)
    word = re.sub(r"ing$", "", word)
    word = re.sub(r"ed$", "", word)
    return word

for word in ["playing", "played", "happily", "running"]:
    print(f"{word:10} -> {simple_suffix_normalize(word)}")

playing    -> play
played     -> play
happily    -> happy
running    -> runn


### Practice 4 — Combine a dictionary with regex rules

Irregular forms should be checked before general suffix rules. This pattern—specific exceptions first, general rules second—is common in rule-based NLP systems.

把单词尽量还原成基础形式，并优先处理不规则变化。

In [ ]:
irregular_forms = {
    "went": "go", "seen": "see", "ran": "run",
    "written": "write", "took": "take", "bought": "buy",
    "drank": "drink", "swam": "swim", "chose": "choose", "flew": "fly",
}

def normalize_with_exceptions(word, exceptions):
    normalized = word.lower()

    # 检查这个单词是不是在不规则词典中
    # 如果在，就直接返回对应的原形。
    # 如果不存在，就交给simple_suffix_normalize()还原成原型

    if normalized in exceptions:
        return exceptions[normalized]
    return simple_suffix_normalize(normalized)

for word in ["playing", "played", "happily", "took", "swam"]:
    print(f"{word:10} -> {normalize_with_exceptions(word, irregular_forms)}")   # 取每个单词，都用normalize_with_exceptions()还原成原型

playing    -> play
played     -> play
happily    -> happy
took       -> take
swam       -> swim


#### TRY小挑战
Write a function `extract_log_fields(log_lines)` that:

1. uses the named-group log pattern from Example 4;
2. skips lines that do not match;
3. returns a list of dictionaries; and
4. converts the result into a useful summary, such as counts by log level or by user.

Test it on at least four lines, including one malformed line.

In [23]:
sample_logs = [
    "2026-09-08 INFO user=alice action=login",
    "2026-09-08 ERROR user=bob action=upload",
    "malformed line",
    "2026-09-09 WARNING user=alice action=retry",
]

log_pattern = (
    r"(?P<date>\d{4}-\d{2}-\d{2})\s+"
    r"(?P<level>INFO|WARNING|ERROR)\s+"
    r"user=(?P<user>\w+)\s+"
    r"action=(?P<action>\w+)"
)

def extract_log_fields(log_lines):
    records = []
    for line in log_lines:
        match = re.fullmatch(log_pattern, line)
        print(type(match))
        if match:
            records.append(match.groupdict())
    return records

records = extract_log_fields(sample_logs)
records

<class 're.Match'>
<class 're.Match'>
<class 'NoneType'>
<class 're.Match'>


[{'date': '2026-09-08', 'level': 'INFO', 'user': 'alice', 'action': 'login'},
 {'date': '2026-09-08', 'level': 'ERROR', 'user': 'bob', 'action': 'upload'},
 {'date': '2026-09-09',
  'level': 'WARNING',
  'user': 'alice',
  'action': 'retry'}]